In [ ]:
# BLOCK 1: Build arena mask + mappings (modular arena)
# You can change rows, cols, and corridor params below to create different arenas.

def make_two_rooms_with_corridor(rows=3, left_cols=2, corridor_cols=3, right_cols=2,
                                 corridor_rows=(1),  # tuple/list of row indices that ARE passable in corridor
                                 prefer_total_cols=None):
    """
    Construct a boolean mask for two square rooms (left, right) connected by a corridor.
    Returns mask (rows x total_cols) with True for passable cells and False for blocked.
    corridor_rows: which rows (0-indexed) in the corridor region are allowed to be passable.
                   e.g., (1,2) for two middle rows in a 4-row grid.
    prefer_total_cols: if you need a target total width (e.g. 10), you can supply it and
                       function will try to center corridor; if None, uses left+corridor+right.
    """
    # compute total_cols
    default_total = left_cols + corridor_cols + right_cols
    if prefer_total_cols is None:
        total_cols = default_total
    else:
        total_cols = prefer_total_cols
        # If prefer_total_cols differs from default, we will embed the three regions appropriately
        # We'll place left at col 0, corridor after left, and right at the end so overall fits.
    mask = np.zeros((rows, total_cols), dtype=bool)

    # left room: columns 0 .. left_cols-1 (all rows)
    left_start = 0
    left_end = left_start + left_cols  # exclusive
    mask[:, left_start:left_end] = True

    # corridor: place immediately after left
    corridor_start = left_end
    corridor_end = corridor_start + corridor_cols
    if corridor_end > total_cols:
        raise ValueError("Corridor doesn't fit in prefer_total_cols; increase total width or reduce corridor width.")
    # corridor rows: only allow rows in corridor_rows list (e.g., middle rows)
    for r in corridor_rows:
        mask[r, corridor_start:corridor_end] = True

    # right room: place at the end to make total width sensible
    right_end = total_cols
    right_start = right_end - right_cols
    mask[:, right_start:right_end] = True

    return mask, {"left": (slice(None), slice(left_start, left_end)),
                  "corridor": (slice(None), slice(corridor_start, corridor_end)),
                  "right": (slice(None), slice(right_start, right_end))}

# Example 1: two 4x4 rooms connected by 2x3 corridor => total cols = 4 + 3 + 4 = 11
rows = 2
left_cols = 2
corridor_cols = 2   # corridor width
right_cols = 2
corridor_rows = (1,)  # allow rows 1 and 2 (middle two rows), top(0) and bottom(3) are blocked in corridor
mask, regions = make_two_rooms_with_corridor(rows=rows, left_cols=left_cols,
                                             corridor_cols=corridor_cols, right_cols=right_cols,
                                             corridor_rows=corridor_rows, prefer_total_cols=None)

# If you *really* want a 4x10 grid, you can call with prefer_total_cols=10, but ensure corridor fits:
# mask, regions = make_two_rooms_with_corridor(rows=4, left_cols=4, corridor_cols=2, right_cols=4,
#                                              corridor_rows=(1,2), prefer_total_cols=10)

print("Arena mask shape:", mask.shape)
print("Passable cells count:", mask.sum())


In [ ]:
# BLOCK 2: Compact mapping from (r,c) -> state index and back for only passable cells


rows, cols = mask.shape
# create list of valid (r,c) and mapping arrays
valid_rc = [(r, c) for r in range(rows) for c in range(cols) if mask[r, c]]
n_states = len(valid_rc)


# observation modalities -> threat intensity/dist, shelter, agent location
n_threat_obs = 4                         # intensities 0..3 with higher intensity is closer => can make this modular in the future
n_shelter_obs = 2 # [NOT AT, AT]
n_agent_obs = n_states

actions = ["up", "down", "left", "right", "stay"]
n_actions = len(actions)

# maps: rc -> compact state id (0..n_states-1), and state->(r,c)
rc_to_state = -np.ones((rows, cols), dtype=int)   # -1 for blocked cells
state_to_rc = [None] * n_states
for i, (r, c) in enumerate(valid_rc):
    rc_to_state[r, c] = i
    state_to_rc[i] = (r, c)

def state_idx_to_rc(idx):
    return state_to_rc[int(idx)]

def rc_to_state_idx(r, c):
    return int(rc_to_state[r, c])

print("n_states (valid cells):", n_states)
print("Sample mapping: first 6 states -> rc:", state_to_rc[:6])


In [ ]:
# BLOCK 3: step_from using neighbor adjacency on masked arena
# actions defined as before: 0=up,1=down,2=left,3=right,4=stay

def step_from_state(state_idx, action):
    r, c = state_idx_to_rc(state_idx)
    if action == 0:   # up
        nr, nc = r - 1, c
    elif action == 1: # down
        nr, nc = r + 1, c
    elif action == 2: # left
        nr, nc = r, c - 1
    elif action == 3: # right
        nr, nc = r, c + 1
    elif action == 4: # stay
        nr, nc = r, c
    else:
        raise ValueError("bad action")

    # If target is outside bounds or blocked, stay in place
    if nr < 0 or nr >= rows or nc < 0 or nc >= cols:
        return state_idx
    if not mask[nr, nc]:
        return state_idx
    return rc_to_state_idx(nr, nc)

# Example sanity checks
for i in range(min(6, n_states)):
    r,c = state_idx_to_rc(i)
    print("state", i, "rc", (r,c), "neighbors:", [step_from_state(i,a) for a in range(len(actions))])


In [ ]:
def render_grid_frame_arena(agent_state, threat_state, shelter_state, visited_states, step,
                            threat_posterior=None, cell_size=48):
    """
    Draw arena, then overlay threat posterior heatmap (semi-transparent),
    then redraw:
      - shelter cells (green),
      - true threat cell (pink with bold outline),
      - agent (blue, top-most).
    This ensures the true threat cell is a single stable visible marker even when the posterior
    colors vary across the last column.
    shelter_state may be a single int or an iterable of ints.
    """
    # normalize shelter_state to a set for membership tests
    if isinstance(shelter_state, (list, tuple, np.ndarray, set)):
        shelter_set = set(int(x) for x in shelter_state)
    else:
        shelter_set = {int(shelter_state)}

    W = cols * cell_size
    H = rows * cell_size
    from PIL import Image, ImageDraw
    img = Image.new('RGB', (W, H), (255,255,255))
    draw = ImageDraw.Draw(img)

    # 1) Draw base grid: blocked cells, visited (light gray), empty cells white.
    for r in range(rows):
        for c in range(cols):
            x0 = c * cell_size
            y0 = r * cell_size
            x1 = x0 + cell_size - 1
            y1 = y0 + cell_size - 1
            if not mask[r, c]:
                fill = (50,50,50)   # blocked cell (dark)
            else:
                idx = rc_to_state_idx(r, c)
                if idx in visited_states and idx not in shelter_set and idx != agent_state and idx != threat_state:
                    fill = (220,220,220)  # visited but not special
                else:
                    fill = (255,255,255)  # default floor
            draw.rectangle([x0, y0, x1, y1], fill=fill, outline=(0,0,0))

    # 2) Overlay heatmap (posterior) as semi-transparent red rectangles (if provided)
    if threat_posterior is not None:
        import numpy as _np
        heat = _np.zeros((rows, cols))
        for s_idx in range(len(threat_posterior)):
            r,c = state_idx_to_rc(s_idx)
            heat[r, c] = float(threat_posterior[s_idx])
        # apply overlay AFTER base drawing
        for r in range(rows):
            for c in range(cols):
                if mask[r, c] and heat[r,c] > 0:
                    x0 = c * cell_size
                    y0 = r * cell_size
                    # scale alpha gently so small probs are faint
                    alpha = min(0.9, float(heat[r,c]) * 2.5)
                    overlay = Image.new('RGBA', (cell_size, cell_size), (255,0,0,int(alpha*200)))
                    img.paste(overlay, (x0, y0), overlay)

    # 3) Redraw shelter cells (green) on top of heatmap so shelter region remains visible
    for s_idx in shelter_set:
        r, c = state_idx_to_rc(s_idx)
        x0 = c * cell_size
        y0 = r * cell_size
        x1 = x0 + cell_size - 1
        y1 = y0 + cell_size - 1
        draw.rectangle([x0, y0, x1, y1], fill=(150,255,150), outline=(0,0,0), width=1)

    # 4) Redraw true threat cell (single) — pink with bold outline so it's immediately obvious
    if threat_state is not None:
        tr = int(threat_state)
        r_t, c_t = state_idx_to_rc(tr)
        x0 = c_t * cell_size
        y0 = r_t * cell_size
        x1 = x0 + cell_size - 1
        y1 = y0 + cell_size - 1
        # pink fill slightly translucent look (we just overpaint)
        draw.rectangle([x0, y0, x1, y1], fill=(255,150,150), outline=(0,0,0), width=2)
        # add a small 'T' label in the center (optional) to further disambiguate
        try:
            # center text roughly
            txt_x = x0 + cell_size // 2 - 6
            txt_y = y0 + cell_size // 2 - 8
            draw.text((txt_x, txt_y), "T", fill=(0,0,0))
        except Exception:
            pass  # if font/drawing fails, ignore

    # 5) Redraw agent ON TOP so it's always visible (blue)
    if agent_state is not None:
        ag = int(agent_state)
        r_a, c_a = state_idx_to_rc(ag)
        x0 = c_a * cell_size
        y0 = r_a * cell_size
        x1 = x0 + cell_size - 1
        y1 = y0 + cell_size - 1
        draw.rectangle([x0, y0, x1, y1], fill=(30,30,200), outline=(0,0,0), width=2)
        try:
            txt_x = x0 + cell_size // 2 - 8
            txt_y = y0 + cell_size // 2 - 8
            draw.text((txt_x, txt_y), "A", fill=(255,255,255))
        except Exception:
            pass

    return np.array(img)

In [ ]:
# ====== BEGIN: Continuous Active Inference agent + visualization edits ======
import math
import numpy as np

# --- Parameters: coordinate conventions ---
# We'll use continuous coordinates in "cell units": cell (r,c) center is (c+0.5, r+0.5)
cell_unit = 1.0
cell_center = lambda r,c: np.array([c + 0.5, r + 0.5])  # (x,y) order: x across cols, y across rows

# Active Inference controller params
dt = 0.05                 # integration timestep (s)
gamma_v = 1.5             # damping on speed
K_mu = 5.0                # belief update gain (multiplies sensory correction)
prec_pos = 30.0           # precision of position observations
prec_vel = 20.0           # precision of speed (proprioception)
k_p = 1.8                 # proportional gain mapping position error to desired speed
v_max = 1.2               # max forward speed (cell units / s)
k_v = 10.0                # speed control gain -> acceleration
k_omega = 6.0             # heading control gain -> angular velocity

# small agent size relative to cell
agent_radius = 0.22 * cell_unit  # radius in same units (cell units)
agent_draw_radius_px = int(max(4, agent_radius * 48))  # for drawing with default cell_size=48

# --- initialize real (physical) continuous state of the agent ---
# real_state is dict: x (2D), theta (rad), v (scalar)
start_state_cell = state_to_rc[0]  # pick first valid state as start
start_x = np.array([start_state_cell[1] + 0.5, start_state_cell[0] + 0.5])  # (x,y) in cell units
real_state = {"x": start_x.copy(), "theta": 0.0, "v": 0.0}

# --- initialize beliefs (posterior means) ---
mu = {"x": real_state["x"].copy(), "theta": real_state["theta"], "v": real_state["v"]}

# --- specify the target (center of a target cell) ---
# Example: use the last valid state as target (you can set whichever)
target_state_idx = n_states - 1
target_rc = state_idx_to_rc(target_state_idx)
target_xy = cell_center(*target_rc)  # (x,y) in cell units

# helper: wrap to [-pi,pi]
def wrap_pi(angle):
    return ((angle + np.pi) % (2 * np.pi)) - np.pi

# inference + control step
def ai_step(real_state, mu, target_xy, dt=dt,
            prec_pos=prec_pos, prec_vel=prec_vel,
            K_mu=K_mu):
    """
    One inference + control step:
      - observe (noisy) position and speed,
      - update belief mu by simple gradient descent on prediction errors,
      - compute motor commands (a, omega) to reduce position & heading error,
      - apply motor commands to the real_state (Euler integration).
    Returns updated (real_state, mu) and action (a,omega).
    """
    # 1) Simulate noisy observations (in real scenario, you'd get sensors)
    obs_pos = real_state["x"] + np.random.normal(scale=0.02, size=2)  # small sensor noise
    obs_v = real_state["v"] + np.random.normal(scale=0.01)             # proprio noise

    # 2) Prediction errors (observation - predicted)
    e_pos = obs_pos - mu["x"]     # 2-vector
    e_v = obs_v - mu["v"]

    # 3) Belief dynamics f(mu,u) (prior motion model) -- here we use simple kinematics
    mu_dot = {}
    mu_dot["x"] = np.array([mu["v"] * math.cos(mu["theta"]), mu["v"] * math.sin(mu["theta"])])
    mu_dot["theta"] = 0.0  # we let observations and control influence theta via correction
    mu_dot["v"] = -gamma_v * mu["v"]  # prior damping (no commanded accel in prior)

    # 4) Sensory correction term: J^T * precision * error
    # Observation jacobian J w.r.t mu for pos is identity on x (2x2), for v is 1.
    corr_x = K_mu * (prec_pos * e_pos)   # 2-vector
    corr_v = K_mu * (prec_vel * e_v)     # scalar
    # For theta we have no direct exteroceptive observation; we will infer heading from positional error:
    # approximate correction for theta: push mu_theta towards direction of observed velocity/position change
    # compute desired heading from obs_pos - mu_x (small step)
    desired_heading = math.atan2(target_xy[1] - mu["x"][1], target_xy[0] - mu["x"][0])
    e_theta = wrap_pi(desired_heading - mu["theta"])
    corr_theta = K_mu * 0.5 * e_theta

    # 5) Euler update of beliefs
    mu["x"] = mu["x"] + dt * (mu_dot["x"] + corr_x)
    mu["theta"] = wrap_pi(mu["theta"] + dt * (mu_dot["theta"] + corr_theta))
    mu["v"] = float(mu["v"] + dt * (mu_dot["v"] + corr_v))

    # 6) Decide desired translational speed and heading (simple mapping from pos error)
    pos_err_vec = (target_xy - mu["x"])
    dist = np.linalg.norm(pos_err_vec)
    # desired speed proportional to distance, clipped
    v_desired = min(v_max, k_p * dist)
    # desired heading towards target (use belief)
    theta_desired = math.atan2(pos_err_vec[1], pos_err_vec[0])
    theta_err = wrap_pi(theta_desired - mu["theta"])

    # 7) Motor commands (interpretable as action minimizing proprioceptive errors)
    a_cmd = k_v * (v_desired - mu["v"])   # acceleration command
    omega_cmd = k_omega * theta_err      # angular velocity command

    # small saturations
    a_cmd = float(np.clip(a_cmd, -5.0, 5.0))
    omega_cmd = float(np.clip(omega_cmd, -6.0, 6.0))

    # 8) Apply action to real state (simple physics / Euler integrate)
    # velocity dynamics
    real_state["v"] = float(real_state["v"] + dt * (-gamma_v * real_state["v"] + a_cmd))
    # heading
    real_state["theta"] = wrap_pi(real_state["theta"] + dt * omega_cmd)
    # position
    real_state["x"] = real_state["x"] + dt * real_state["v"] * np.array([math.cos(real_state["theta"]), math.sin(real_state["theta"])])
    # ensure we don't leave the arena bounds: clip to [0, cols]x[0, rows] and reflect off blocked cells
    # convert pos to nearest cell rc to test mask
    cx, cy = real_state["x"][0], real_state["x"][1]  # x is columns, y is rows
    # Clip to grid extents (prevents leaving the drawing area)
    cx = float(np.clip(cx, 0.001, cols - 0.001))
    cy = float(np.clip(cy, 0.001, rows - 0.001))
    real_state["x"] = np.array([cx, cy])

    # If the center lies in a blocked cell, push back to nearest passable cell center (simple safety)
    cell_r = int(math.floor(real_state["x"][1]))
    cell_c = int(math.floor(real_state["x"][0]))
    if cell_r < 0 or cell_r >= rows or cell_c < 0 or cell_c >= cols or not mask[cell_r, cell_c]:
        # snap to nearest passable state's center
        # compute nearest valid cell center (euclidean in cell coords)
        dmin = 1e9
        best_xy = real_state["x"].copy()
        for s_idx in range(n_states):
            r_s, c_s = state_idx_to_rc(s_idx)
            xy = cell_center(r_s, c_s)
            d = np.linalg.norm(xy - real_state["x"])
            if d < dmin:
                dmin = d; best_xy = xy
        real_state["x"] = best_xy
        real_state["v"] = 0.0  # stop when colliding

    return real_state, mu, (a_cmd, omega_cmd), dist

# --- modify rendering function to draw continuous agent + heading + target ---
from PIL import Image, ImageDraw, ImageFont

def render_grid_frame_arena_continuous(real_state, mu, target_xy, threat_state, shelter_state, visited_states, step,
                                       threat_posterior=None, cell_size=48):
    """
    Builds on your original render function but:
      - draws agent as a circle smaller than a cell at continuous position (real_state['x'])
      - draws a heading arrow from the agent
      - draws the target as a small filled circle (red), and also shows belief mu as faint marker (optional)
    """
    # reuse a lot of original code for grid + heatmap + shelter + threat
    W = cols * cell_size
    H = rows * cell_size
    img = Image.new('RGB', (W, H), (255,255,255))
    draw = ImageDraw.Draw(img)

    # 1) base grid drawing (blocked, visited, etc.)
    for r in range(rows):
        for c in range(cols):
            x0 = c * cell_size
            y0 = r * cell_size
            x1 = x0 + cell_size - 1
            y1 = y0 + cell_size - 1
            if not mask[r, c]:
                fill = (50,50,50)
            else:
                idx = rc_to_state_idx(r, c)
                if idx in visited_states and idx not in (shelter_state if isinstance(shelter_state,(list,tuple,set)) else {shelter_state}) \
                   and idx != threat_state:
                    fill = (220,220,220)
                else:
                    fill = (255,255,255)
            draw.rectangle([x0, y0, x1, y1], fill=fill, outline=(0,0,0))

    # 2) heatmap overlay if present
    if threat_posterior is not None:
        heat = np.zeros((rows, cols))
        for s_idx in range(len(threat_posterior)):
            r,c = state_idx_to_rc(s_idx)
            heat[r, c] = float(threat_posterior[s_idx])
        for r in range(rows):
            for c in range(cols):
                if mask[r,c] and heat[r,c] > 0:
                    x0 = c * cell_size
                    y0 = r * cell_size
                    alpha = min(0.9, float(heat[r,c]) * 2.5)
                    overlay = Image.new('RGBA', (cell_size, cell_size), (255,0,0,int(alpha*200)))
                    img.paste(overlay, (x0, y0), overlay)

    # 3) Shelter region
    if isinstance(shelter_state, (list, tuple, np.ndarray, set)):
        shelter_set = set(int(x) for x in shelter_state)
    else:
        shelter_set = {int(shelter_state)}
    for s_idx in shelter_set:
        r,c = state_idx_to_rc(s_idx)
        x0 = c * cell_size; y0 = r * cell_size
        x1 = x0 + cell_size - 1; y1 = y0 + cell_size - 1
        draw.rectangle([x0, y0, x1, y1], fill=(150,255,150), outline=(0,0,0), width=1)

    # 4) threat cell
    if threat_state is not None:
        tr = int(threat_state)
        r_t, c_t = state_idx_to_rc(tr)
        x0 = c_t * cell_size; y0 = r_t * cell_size
        x1 = x0 + cell_size - 1; y1 = y0 + cell_size - 1
        draw.rectangle([x0, y0, x1, y1], fill=(255,150,150), outline=(0,0,0), width=2)
        try:
            txt_x = x0 + cell_size // 2 - 6
            txt_y = y0 + cell_size // 2 - 8
            draw.text((txt_x, txt_y), "T", fill=(0,0,0))
        except Exception:
            pass

    # 5) target marker (continuous) - small red circle at target_xy (in cell units -> pixels)
    tx_px = int(target_xy[0] * cell_size)
    ty_px = int(target_xy[1] * cell_size)
    r_tg = max(3, int(0.12 * cell_size))
    draw.ellipse([tx_px - r_tg, ty_px - r_tg, tx_px + r_tg, ty_px + r_tg], fill=(220,50,50), outline=(0,0,0))

    # 6) Belief location (optional faint blue dot)
    mu_px = (int(mu["x"][0] * cell_size), int(mu["x"][1] * cell_size))
    r_mu = max(2, int(0.10 * cell_size))
    draw.ellipse([mu_px[0] - r_mu, mu_px[1] - r_mu, mu_px[0] + r_mu, mu_px[1] + r_mu], fill=(120,180,255,120))

    # 7) Draw agent at continuous real_state position, smaller than cell
    ag_x_px = int(real_state["x"][0] * cell_size)
    ag_y_px = int(real_state["x"][1] * cell_size)
    r_px = max(3, int(agent_radius * cell_size))
    # body
    draw.ellipse([ag_x_px - r_px, ag_y_px - r_px, ag_x_px + r_px, ag_y_px + r_px], fill=(30,30,200), outline=(0,0,0), width=2)
    # heading arrow
    head_len = int(max(8, 0.8 * cell_size * 0.4))
    hx = ag_x_px + head_len * math.cos(real_state["theta"])
    hy = ag_y_px + head_len * math.sin(real_state["theta"])
    draw.line([ag_x_px, ag_y_px, int(hx), int(hy)], fill=(255,255,255), width=2)
    # small "A" label
    try:
        draw.text((ag_x_px - 6, ag_y_px - 6), "A", fill=(255,255,255))
    except Exception:
        pass

    # 8) optionally draw grid step counter
    try:
        draw.text((4, 4), f"step {step}", fill=(0,0,0))
    except Exception:
        pass

    return np.array(img)

# Example usage / small simulator run (simulate a few steps and render frames)
visited = set([0])
frames = []
for t in range(60):  # simulate 60 steps (~3s with dt=0.05)
    real_state, mu, action, dist = ai_step(real_state, mu, target_xy, dt=dt)
    # record visited discrete cell
    rc = (int(math.floor(real_state["x"][1])), int(math.floor(real_state["x"][0])))
    if 0 <= rc[0] < rows and 0 <= rc[1] < cols and mask[rc]:
        visited.add(rc_to_state_idx(rc[0], rc[1]))
    # render
    frame = render_grid_frame_arena_continuous(real_state, mu, target_xy,
                                               threat_state=None, shelter_state=list(shelter_set) if 'shelter_set' in locals() else 0,
                                               visited_states=visited, step=t, threat_posterior=None, cell_size=48)
    frames.append(frame)

# If in a Jupyter notebook, display the last frame inline (requires matplotlib)
try:
    import matplotlib.pyplot as plt
    plt.imshow(frames[-1])
    plt.axis('off')
    plt.show()
except Exception:
    pass

# ====== END: Continuous Active Inference agent + visualization edits ======
